# AgentRegistry, end to end: scaffold a dice agent → run it on kagent **and** AWS Bedrock AgentCore

**Persona:** a **developer**. Run each cell like a terminal command, see the output, and watch the project appear in the Explorer. We scaffold a dice-rolling agent with `arctl`, run it locally, publish it to the catalog, then deploy **the same agent** to two runtimes — Solo Enterprise for **kagent** (local kind) and **AWS Bedrock AgentCore** — by changing one line, the Deployment's `runtimeRef`.

> **Kernel:** pick the **Bash** kernel (top-right). Setup (`./scripts/setup-env.sh` + `./scripts/setup.sh`) is one-time engineer pre-work — see the README. This notebook assumes the platform is already up.

> As you go, open the **`agentdemo/`** folder that appears in the Explorer to walk through the generated code.

## Connect to the platform

Load credentials, put `arctl` on the path, mint a catalog token. `arctl get runtimes` confirms the control plane is live.

In [ ]:
set -a
[ -f .env.local ] && . ./.env.local
[ -n "${SECRETS_FILE:-}" ] && [ -f "$SECRETS_FILE" ] && . "$SECRETS_FILE"
set +a
export PATH="$HOME/.arctl/bin:$PATH"
export CLUSTER_NAME="${CLUSTER_NAME:-agentcore-demo}"
export ARCTL_API_BASE_URL="${ARCTL_API_BASE_URL:-http://localhost:12121}"
export ARCTL_API_TOKEN="$(curl -s -X POST "$ARCTL_API_BASE_URL/api/autoauth/oauth/token" \
  -H 'Content-Type: application/x-www-form-urlencoded' \
  -d 'grant_type=client_credentials&client_id=admin&scope=openid profile email Groups' | jq -r .access_token)"
echo "arctl $(arctl version 2>/dev/null | awk '/arctl version/{print $3}') · token $([ -n "$ARCTL_API_TOKEN" ] && echo ok || echo MISSING)"
arctl get runtimes

## 1. Create a new agent project

`arctl init agent` scaffolds a complete, runnable project — source, manifest (`agent.yaml`), Dockerfile, env wiring, tests. With no custom instruction it generates the **dice-rolling** agent. Run this, then **open `agentdemo/` in the Explorer.**

In [ ]:
rm -rf agentdemo   # clean re-runs
arctl init agent agentdemo --framework adk --language python \
  --model-provider anthropic --model-name claude-haiku-4-5
echo; echo 'Project created — open agentdemo/ in the Explorer:'
find agentdemo -type f -not -path '*/.*' | sort

## 2. Walk through the dice agent

Open `agentdemo/agentdemo/agent.py` in the editor, or print it here. Note the two tools the model can call — `roll_die` and `check_prime` — and the model wiring.

In [ ]:
cat agentdemo/agentdemo/agent.py

## 3. Build and run it locally

`arctl build` builds the agent image from the generated Dockerfile. Then `arctl run` starts it and drops you into an interactive A2A chat — the full agent + tools loop on Docker alone. `run` is interactive, so use a **terminal** for it:

```sh
arctl run ./agentdemo
# then:  Roll a 20-sided die and tell me if the result is prime.
# Ctrl-C to exit
```

In [ ]:
arctl build ./agentdemo

## 4. Publish to the catalog

`arctl build --push` pushes the image to the local registry; `arctl apply` registers the agent in the catalog so it can be deployed by reference.

In [ ]:
arctl build ./agentdemo --push          # -> localhost:5001/agentdemo:latest
arctl apply -f agentdemo/agent.yaml
arctl get agent agentdemo

## 5. Point the registry at the cluster (Kubernetes **Runtime**)

A `Runtime` tells the registry where agents can run. The daemon runs inside Docker, so we join it to the kind network and hand it the cluster's **internal** kubeconfig.

In [ ]:
DAEMON_CTR=$(docker ps --filter publish=12121 --format '{{.Names}}' | head -1)
docker network connect kind "$DAEMON_CTR" 2>/dev/null || true
cat > /tmp/runtime-kagent.yaml <<EOF
apiVersion: ar.dev/v1alpha1
kind: Runtime
metadata:
  name: kind-kagent
spec:
  type: Kubernetes
  config:
    namespace: kagent
    kubeconfig: |
$(kind get kubeconfig --internal --name "$CLUSTER_NAME" | sed 's/^/      /')
EOF
arctl apply -f /tmp/runtime-kagent.yaml
arctl get runtimes

## 6. Deploy the agent onto kagent (runtime #1)

A `Deployment` binds the Agent to a Runtime. Note `runtimeRef.name: kind-kagent` — **this is the one line that changes for AWS later.** The registry translates it into kagent CRDs and the controller schedules the pod. The deployment passes `ANTHROPIC_API_KEY` for the model.

In [ ]:
cat > /tmp/deploy-kagent.yaml <<EOF
apiVersion: ar.dev/v1alpha1
kind: Deployment
metadata:
  name: agentdemo
spec:
  targetRef:   { kind: Agent,   name: agentdemo }
  runtimeRef:  { kind: Runtime, name: kind-kagent }
  env:
    ANTHROPIC_API_KEY: "${ANTHROPIC_API_KEY}"
EOF
arctl apply -f /tmp/deploy-kagent.yaml
arctl get deployments

In [ ]:
# Watch it land as a kagent Agent + pod (give the controller a moment):
kubectl --context kind-$CLUSTER_NAME -n kagent get agents,pods | grep -viE 'kmcp|controller|postgres|tools'

## 7. Talk to the dice agent — through real OIDC

The agent sits behind Solo Enterprise for kagent's OIDC interceptor. `ask.sh` mints a real Keycloak token for **alice** (group `field-fte` → kagent Admin), then sends an A2A message. Watch it call `roll_die` then `check_prime`.

In [ ]:
export AGENT=$(kubectl --context kind-$CLUSTER_NAME -n kagent get agents.kagent.dev -o name 2>/dev/null | sed 's#.*/##' | grep -i agentdemo | head -1)
./scripts/ask.sh "Roll a 20-sided die and tell me whether the result is a prime number."

Optional — open the kagent dashboard: `./scripts/port-forward.sh` then `http://localhost:8080`.

---
# The punchline: the **same** agent on AWS Bedrock AgentCore (runtime #2)

We deploy the identical `agentdemo` to AWS. AgentCore builds the agent from source and runs it on **native Bedrock Claude via the AWS role** (no API key). Two small changes make the one agent multi-cloud: a `BedrockClaude` model adapter, and picking the model from `MODEL_PROVIDER`. **Needs an AWS account; skip for a local-only demo.**

## 8. Make the agent multi-cloud

Drop the `BedrockClaude` adapter into the project and switch `create_model()` to pick the provider from `MODEL_PROVIDER` (Anthropic on kagent, Bedrock on AgentCore). Same agent, both clouds.

In [ ]:
cp templates/bedrock_model.py agentdemo/agentdemo/bedrock_model.py
python3 - <<'PY'
import re, pathlib
p = pathlib.Path('agentdemo/agentdemo/agent.py'); s = p.read_text()
s = s.replace('from google.adk.models.lite_llm import LiteLlm\n', '')
new = '''def create_model():
    """Pick the model from MODEL_PROVIDER: anthropic (kagent, via LiteLLM +
    ANTHROPIC_API_KEY) or bedrock (AgentCore, via the AWS role — no key)."""
    import os
    if os.environ.get("MODEL_PROVIDER", "anthropic").lower() == "bedrock":
        from .bedrock_model import BedrockClaude
        return BedrockClaude(model=os.environ.get("MODEL_NAME", "us.anthropic.claude-haiku-4-5-20251001-v1:0"))
    from google.adk.models.lite_llm import LiteLlm
    return LiteLlm(model=os.environ.get("MODEL_NAME", "anthropic/claude-haiku-4-5"))
'''
s = re.sub(r'def create_model\(\):.*?(?=\n\nroot_agent|\nroot_agent|\Z)', new.rstrip()+'\n', s, count=1, flags=re.S)
p.write_text(s)
pp = pathlib.Path('agentdemo/pyproject.toml'); t = pp.read_text()
if 'anthropic[bedrock]' not in t:
    t = t.replace('dependencies = [', 'dependencies = [\n  "anthropic[bedrock]>=0.40",', 1); pp.write_text(t)
print('patched create_model() + added anthropic[bedrock]; bedrock_model.py vendored')
PY
sed -n '/def create_model/,/LiteLlm(model/p' agentdemo/agentdemo/agent.py

## 9. Sign in to AWS

`aws sso login` (uses `AWS_PROFILE` from `.env.local`). The cell also hands the credentials to the arctl daemon (restarts it) so it can assume the cross-account role when it manages AgentCore.

In [ ]:
if [ -z "${AWS_PROFILE:-}" ]; then echo "Set AWS_PROFILE in .env.local (./scripts/setup-env.sh) and re-run Connect."; else
  aws sts get-caller-identity >/dev/null 2>&1 || aws sso login --profile "$AWS_PROFILE"
  export AWS_REGION="${AWS_REGION:-us-east-1}"
  export AWS_ACCOUNT_ID="$(aws sts get-caller-identity --query Account --output text)"
  echo "AWS session live — account ****${AWS_ACCOUNT_ID: -4} / region $AWS_REGION"
  eval "$(aws configure export-credentials --format env)"
  export AWS_ACCESS_KEY_ID AWS_SECRET_ACCESS_KEY AWS_SESSION_TOKEN
  export DOCKER_REPO="${DOCKER_REPO:-solo-public/agentregistry-enterprise}" OIDC_AUTO_AUTH_ENABLED=true
  arctl daemon stop >/dev/null 2>&1; arctl daemon start >/dev/null 2>&1
  DC=$(docker ps --filter publish=12121 --format '{{.Names}}' | head -1)
  docker network connect kind "$DC" 2>/dev/null || true; sleep 5
  export ARCTL_API_TOKEN="$(curl -s -X POST "$ARCTL_API_BASE_URL/api/autoauth/oauth/token" -H 'Content-Type: application/x-www-form-urlencoded' -d 'grant_type=client_credentials&client_id=admin&scope=openid profile email Groups' | jq -r .access_token)"
  echo 'daemon restarted with AWS credentials'
fi

## 10. Grant AgentRegistry access + register the AgentCore runtime

Generate a CloudFormation cross-account role (no AWS change — just the template + an External ID), deploy it, read the role ARN, then register a `BedrockAgentCore` Runtime.

In [ ]:
mkdir -p .agentcore
arctl runtime setup bedrock-agent-core --aws-account-id "$AWS_ACCOUNT_ID" \
  --role-name AgentRegistryAccessRole-agentcore-demo \
  2> >(tee .agentcore/setup.stderr >&2) > .agentcore/cf.yaml
export AWS_EXTERNAL_ID=$(grep -ioE 'External ID:[[:space:]]*[A-Za-z0-9_-]+' .agentcore/setup.stderr | awk '{print $NF}' | head -1)
if aws cloudformation describe-stacks --stack-name AgentRegistryAccess >/dev/null 2>&1; then echo 'stack exists'; else
  aws cloudformation create-stack --stack-name AgentRegistryAccess --template-body file://.agentcore/cf.yaml --capabilities CAPABILITY_NAMED_IAM
  aws cloudformation wait stack-create-complete --stack-name AgentRegistryAccess; fi
export AWS_ROLE_ARN=$(aws cloudformation describe-stacks --stack-name AgentRegistryAccess --query 'Stacks[0].Outputs[?OutputKey==`RoleArn`].OutputValue' --output text)
cat > /tmp/runtime-aws.yaml <<EOF
apiVersion: ar.dev/v1alpha1
kind: Runtime
metadata:
  name: aws-agentcore
spec:
  type: BedrockAgentCore
  config: { roleArn: "${AWS_ROLE_ARN}", externalId: "${AWS_EXTERNAL_ID}", region: "${AWS_REGION}" }
EOF
arctl apply -f /tmp/runtime-aws.yaml
arctl get runtimes

## 11. Push the agent to ECR + git, point the Agent at them

AgentCore can't pull `localhost:5001` and clones the source from git at deploy, so we push the image to **ECR** and the project to your repo (`AGENT_GIT_URL`), then re-publish the Agent as `modelProvider: bedrock`.

In [ ]:
export ECR_HOST="${AWS_ACCOUNT_ID}.dkr.ecr.${AWS_REGION}.amazonaws.com"
export ECR_IMAGE="${ECR_HOST}/agentdemo:0.0.1"
aws ecr describe-repositories --repository-names agentdemo >/dev/null 2>&1 || aws ecr create-repository --repository-name agentdemo >/dev/null
aws ecr get-login-password --region "$AWS_REGION" | docker login --username AWS --password-stdin "$ECR_HOST" >/dev/null
arctl build ./agentdemo --push --platform linux/amd64 --image "$ECR_IMAGE"

# Push the (now multi-cloud) agentdemo project to your repo for AgentCore to clone.
: "${AGENT_GIT_URL:?set AGENT_GIT_URL in .env.local (./scripts/setup-env.sh)}"
SLUG="${AGENT_GIT_URL#https://github.com/}"; SLUG="${SLUG%.git}"
PUSH_URL="$AGENT_GIT_URL"
command -v gh >/dev/null 2>&1 && PUSH_URL="https://x-access-token:$(gh auth token)@github.com/${SLUG}.git"
T=$(mktemp -d); cp -R agentdemo "$T/agentdemo"
( cd "$T" && git init -qb "${AGENT_GIT_BRANCH:-main}" && git add -A \
  && git -c user.email=demo@local -c user.name=demo commit -qm 'agentdemo source' \
  && git remote add origin "$PUSH_URL" && git push -fq origin "${AGENT_GIT_BRANCH:-main}" ) && echo 'pushed agentdemo source'
rm -rf "$T"

CLONE_URL="$AGENT_GIT_URL"
[ "$(gh repo view "$SLUG" --json isPrivate -q .isPrivate 2>/dev/null)" = true ] && CLONE_URL="https://x-access-token:$(gh auth token)@github.com/${SLUG}.git"
cat > /tmp/agent-aws.yaml <<EOF
apiVersion: ar.dev/v1alpha1
kind: Agent
metadata: { name: agentdemo }
spec:
  description: Dice-rolling agent (roll_die, check_prime).
  modelName: us.anthropic.claude-haiku-4-5-20251001-v1:0
  modelProvider: bedrock
  source:
    image: ${ECR_IMAGE}
    repository: { url: ${CLONE_URL}, branch: ${AGENT_GIT_BRANCH:-main}, subfolder: agentdemo }
EOF
arctl apply -f /tmp/agent-aws.yaml && echo 'agent re-published (bedrock + ECR + git source)'

## 12. Deploy the same agent to AgentCore (runtime #2)

Same Agent — only `runtimeRef` differs (`aws-agentcore`). `MODEL_PROVIDER=bedrock` makes it use native Bedrock Claude via the AWS role. The cell waits for AWS to provision (CREATING → READY).

In [ ]:
cat > /tmp/deploy-aws.yaml <<EOF
apiVersion: ar.dev/v1alpha1
kind: Deployment
metadata: { name: agentdemo-agentcore }
spec:
  targetRef:  { kind: Agent,   name: agentdemo }
  runtimeRef: { kind: Runtime, name: aws-agentcore }   # <-- the only meaningful change
  runtimeConfig: { region: ${AWS_REGION} }
  env: { MODEL_PROVIDER: bedrock, AWS_REGION: ${AWS_REGION} }
EOF
arctl apply -f /tmp/deploy-aws.yaml
echo 'waiting for the AWS runtime (CREATING -> READY, a few minutes)...'
for i in $(seq 1 25); do
  S=$(aws bedrock-agentcore-control list-agent-runtimes --region "$AWS_REGION" 2>/dev/null | jq -r '.agentRuntimes[]?|select(.agentRuntimeName=="agentdemo_agentcore")|.status')
  echo "[$i] agentdemo_agentcore: ${S:-<none>}"; echo "$S" | grep -qiE 'READY|FAILED' && break; sleep 30
done

## 13. Test the dice agent on AgentCore

Invoke the AWS-hosted runtime directly and see it roll the die — same agent, now answering from AWS.

In [ ]:
ARN=$(aws bedrock-agentcore-control list-agent-runtimes --region "$AWS_REGION" | jq -r '.agentRuntimes[]?|select(.agentRuntimeName=="agentdemo_agentcore")|.agentRuntimeArn')
PAYLOAD='{"jsonrpc":"2.0","id":"r1","method":"message/send","params":{"message":{"role":"user","messageId":"m1","parts":[{"kind":"text","text":"Roll a 20-sided die and tell me whether the result is prime."}]}}}'
aws bedrock-agentcore invoke-agent-runtime --region "$AWS_REGION" --cli-binary-format raw-in-base64-out \
  --agent-runtime-arn "$ARN" --content-type application/json --accept application/json \
  --payload "$PAYLOAD" /tmp/ac-out.json >/dev/null
python3 - <<'PY'
import json
d=json.load(open('/tmp/ac-out.json')); seen=[]
def w(o):
    if isinstance(o,dict):
        if o.get('role')=='user': return
        if o.get('kind')=='text' and isinstance(o.get('text'),str):
            t=o['text'].strip()
            if t and t not in seen: seen.append(t)
        [w(v) for v in o.values()]
    elif isinstance(o,list): [w(v) for v in o]
print('ERROR:', json.dumps(d['error'])) if 'error' in d else (w(d) or print('\n\n'.join(seen) if seen else json.dumps(d)[:1200]))
PY

**The takeaway:** one agent, scaffolded with `arctl`, published once, ran unchanged on Kubernetes (kagent) *and* AWS Bedrock AgentCore. Moving or multi-homing it is a one-line `runtimeRef` change.

## Teardown

```sh
./scripts/cleanup.sh agentcore   # AWS only
./scripts/cleanup.sh             # everything
```

In [ ]:
# Uncomment to tear everything down:
# ./scripts/cleanup.sh